In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/train.csv')
df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [6]:
# Extraer el Deck (primera letra de la cabina) ANTES de perder esa información
df['Deck'] = df['Cabin'].str[0]
df['Deck'] = df['Deck'].fillna('Unknown')

# Mantener también HasCabin (puede seguir siendo útil como variable adicional)
df['HasCabin'] = df['Cabin'].notnull().astype(int)

# Ahora sí, eliminar la columna original
df = df.drop(columns=['Cabin'])

df['Deck'].value_counts()

Deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64

In [7]:
# Agrupar la categoría "T" (solo 1 caso) por ser demasiado rara para generalizar
df['Deck'] = df['Deck'].replace('T', 'Unknown')

df['Deck'].value_counts()

Deck
Unknown    688
C           59
B           47
D           33
E           32
A           15
F           13
G            4
Name: count, dtype: int64

In [8]:
# Rellenar Age con la mediana según Pclass y Sex (más preciso que la mediana general)
df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

# Verificar que ya no queden nulos
df['Age'].isnull().sum()

np.int64(0)

In [9]:
# Rellenar con el puerto más frecuente
moda_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(moda_embarked)

df['Embarked'].isnull().sum()

np.int64(0)

In [10]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Variable adicional útil: viajaba solo o no
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

In [11]:
# Extraer el título del nombre (antes de eliminar la columna Name)
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')

# Ver qué títulos salieron y cuántos hay de cada uno
df['Title'].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [12]:
# Crear bins de edad y tarifa
df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], labels=['Niño', 'Adolescente', 'Adulto', 'Adulto_mayor', 'Anciano'])
df['FareBin'] = pd.qcut(df['Fare'], 4, labels=['Baja', 'Media', 'Alta', 'MuyAlta'])

df[['Age', 'AgeBin', 'Fare', 'FareBin']].head()

,Age,AgeBin,Fare,FareBin
0,22.0,Adulto,7.2500,Baja
1,38.0,Adulto_mayor,71.2833,MuyAlta
2,26.0,Adulto,7.9250,Media
3,35.0,Adulto,53.1000,MuyAlta
4,35.0,Adulto,8.0500,Media


In [13]:
# Agrupar títulos poco frecuentes en una categoría "Rare"
title_map = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Capt': 'Rare', 'Sir': 'Rare'
}
df['Title'] = df['Title'].map(title_map)
df['Title'].value_counts()

Title
Mr        517
Miss      185
Mrs       125
Master     40
Rare       22
Name: count, dtype: int64

In [14]:
# Name y Ticket son texto libre con poco valor predictivo directo
# PassengerId es solo un identificador
df = df.drop(columns=['Name', 'Ticket', 'PassengerId'])

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Deck,HasCabin,FamilySize,IsAlone,Title,AgeBin,FareBin
0,0,3,male,22.0,1,0,7.2500,S,Unknown,0,2,0,Mr,Adulto,Baja
1,1,1,female,38.0,1,0,71.2833,C,C,1,2,0,Mrs,Adulto_mayor,MuyAlta
2,1,3,female,26.0,0,0,7.9250,S,Unknown,0,1,1,Miss,Adulto,Media
3,1,1,female,35.0,1,0,53.1000,S,C,1,2,0,Mrs,Adulto,MuyAlta
4,0,3,male,35.0,0,0,8.0500,S,Unknown,0,1,1,Mr,Adulto,Media


In [15]:
# Rellenar los valores nulos de Title con "Rare" 
df['Title'] = df['Title'].fillna('Rare')


In [16]:
# drop a "age" column since we have AgeBin
df = df.drop(columns=['Age'])

In [17]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
SibSp         0
Parch         0
Fare          0
Embarked      0
Deck          0
HasCabin      0
FamilySize    0
IsAlone       0
Title         0
AgeBin        0
FareBin       0
dtype: int64

In [18]:
import os

# Crear la carpeta si no existe
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/train_clean.csv', index=False)
print("Dataset limpio guardado correctamente ✅")

Dataset limpio guardado correctamente ✅


In [19]:
df.head(5)

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked,Deck,HasCabin,FamilySize,IsAlone,Title,AgeBin,FareBin
0,0,3,male,1,0,7.2500,S,Unknown,0,2,0,Mr,Adulto,Baja
1,1,1,female,1,0,71.2833,C,C,1,2,0,Mrs,Adulto_mayor,MuyAlta
2,1,3,female,0,0,7.9250,S,Unknown,0,1,1,Miss,Adulto,Media
3,1,1,female,1,0,53.1000,S,C,1,2,0,Mrs,Adulto,MuyAlta
4,0,3,male,0,0,8.0500,S,Unknown,0,1,1,Mr,Adulto,Media
